# Imports & loading

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

sys.path.append(str(Path("..").resolve()))
from src.utils import DATA_PROC, REPORTS, get_logger
from src.explain import (
    load_artifacts, compute_shap_values,
    plot_summary, plot_dependence,
    explain_single_borrower, plot_borrower_explanation
)

logger = get_logger("explainability_notebook")

train    = pd.read_parquet(DATA_PROC / "train_features.parquet")
y        = train["TARGET"].copy()
X        = train.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore")

model, feature_names = load_artifacts()
X.columns = feature_names   # ensure alignment
logger.info(f"Loaded model and {len(feature_names)} features")

# Compute SHAP values

In [ ]:
shap_values, X_sample, explainer = compute_shap_values(model, X, sample_size=2000)
print(f"SHAP values shape : {shap_values.shape}")
print(f"Mean |SHAP| (top 5):")
mean_abs = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_sample.columns
).sort_values(ascending=False)
print(mean_abs.head(5).round(4).to_string())

# Global summary plot

In [ ]:
plot_summary(shap_values, X_sample, top_n=20)

# Dependence plots for top 3 features

In [ ]:
for feature in ["EXT_SOURCE_MEAN", "CREDIT_TERM", "ANNUITY_INCOME_RATIO"]:
    plot_dependence(shap_values, X_sample, feature)

# Explain three individual borrowers

In [ ]:
import json

# Pick a high-risk, medium-risk, and low-risk borrower from the sample
probs = model.predict_proba(X_sample)[:, 1]
high_risk_idx = int(np.argmax(probs))
low_risk_idx  = int(np.argmin(probs))
mid_risk_idx  = int(np.argmin(np.abs(probs - 0.5)))

for label, idx in [("High risk", high_risk_idx),
                   ("Medium risk", mid_risk_idx),
                   ("Low risk", low_risk_idx)]:
    print(f"\n{'='*55}")
    print(f"  {label} borrower (idx={idx})")
    print(f"{'='*55}")
    exp = explain_single_borrower(model, explainer, X_sample, idx)
    plot_borrower_explanation(exp)

    print(f"  Default probability : {exp['default_prob']:.1%}")
    print(f"  Decision            : {exp['decision']}")
    print(f"\n  Top risk drivers:")
    for d in exp["risk_drivers"]:
        print(f"    {d['feature']:35s} val={d['value']:.3f}  shap={d['shap']:+.4f}")
    print(f"\n  Protective factors:")
    for d in exp["protectors"]:
        print(f"    {d['feature']:35s} val={d['value']:.3f}  shap={d['shap']:+.4f}")

# Save SHAP feature ranking

In [ ]:
shap_importance = pd.DataFrame({
    "feature":      X_sample.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

shap_importance.to_csv(DATA_PROC / "shap_importance.csv", index=False)
print("Top 10 features by mean |SHAP|:")
print(shap_importance.head(10).round(4).to_string(index=False))
logger.info("Saved: shap_importance.csv")